# Synthetic Data Generation Using RAGAS - RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow!



- 🤝 BREAKOUT ROOM #1
  1. Use RAGAS to Generate Synthetic Data

- 🤝 BREAKOUT ROOM #2
  1. Load them into a LangSmith Dataset
  2. Evaluate our RAG chain against the synthetic test data
  3. Make changes to our pipeline
  4. Evaluate the modified pipeline

SDG is a critical piece of the puzzle, especially for early iteration! Without it, it would not be nearly as easy to get high quality early signal for our application's performance.

Let's dive in!

**For a walkthrough, visit [Loom](https://www.loom.com/share/e09eb4a4ebea4f99acc553669f968abf?sid=3ee89062-075a-420a-95a4-d5c0b7c58fb8)**

# 🤝 BREAKOUT ROOM #1

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

> NOTE: DO NOT RUN THESE CELLS IF YOU ARE RUNNING THIS NOTEBOOK LOCALLY

In [1]:
#!pip install -qU ragas==0.2.10

In [2]:
#!pip install -qU langchain-community==0.3.14 langchain-openai==0.2.14 unstructured==0.16.12 langgraph==0.2.61 langchain-qdrant==0.2.0

### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.

In [3]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /home/mbudisic/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/mbudisic/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [4]:
import os
import getpass
from dotenv import load_dotenv
import getpass

load_dotenv()
os.environ["LANGCHAIN_TRACING_V2"] = "true"

def set_api_key_if_not_present(key_name, prompt_message):
    if key_name not in os.environ or not os.environ[key_name]:
        os.environ[key_name] = getpass.getpass(prompt_message)

set_api_key_if_not_present("OPENAI_API_KEY", "OpenAI API Key:")
set_api_key_if_not_present("TAVILY_API_KEY", "TAVILY_API_KEY:")
set_api_key_if_not_present("LANGCHAIN_API_KEY", "LANGCHAIN_API_KEY:")

We'll also want to set a project name to make things easier for ourselves.

In [5]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data - and download our webpages which we'll be using for our data today.

These webpages are from [Simon Willison's](https://simonwillison.net/) yearly "AI learnings".

- [2023 Blog](https://simonwillison.net/2023/Dec/31/ai-in-2023/)
- [2024 Blog](https://simonwillison.net/2024/Dec/31/llms-in-2024/)

Let's start by collecting our data into a useful pile!

In [6]:
!mkdir data

/home/mbudisic/Applications/Cursor_9f43194406eb2579c899757921a907b9.AppImage: cannot create directory ‘data’: File exists


In [7]:
!curl https://simonwillison.net/2023/Dec/31/ai-in-2023/ -o data/2023_llms.html

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 31524    0 31524    0     0  58752      0 --:--:-- --:--:-- --:--:-- 58703


In [8]:
!curl https://simonwillison.net/2024/Dec/31/llms-in-2024/ -o data/2024_llms.html

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 70549    0 70549    0     0   106k      0 --:--:-- --:--:-- --:--:--  106k


Next, let's load our data into a familiar LangChain format using the `DirectoryLoader`.

In [9]:
from langchain_community.document_loaders import DirectoryLoader

path = "data/"
loader = DirectoryLoader(path, glob="*.html")
docs = loader.load()

In [10]:
docs

[Document(metadata={'source': 'data/2023_llms.html'}, page_content="Simon Willison’s Weblog\n\nSubscribe\n\nStuff we figured out about AI in 2023\n\n31st December 2023\n\n2023 was the breakthrough year for Large Language Models (LLMs). I think it’s OK to call these AI—they’re the latest and (currently) most interesting development in the academic field of Artificial Intelligence that dates back to the 1950s.\n\nHere’s my attempt to round up the highlights in one place!\n\nLarge Language Models\n\nThey’re actually quite easy to build\n\nYou can run LLMs on your own devices\n\nHobbyists can build their own fine-tuned models\n\nWe don’t yet know how to build GPT-4\n\nVibes Based Development\n\nLLMs are really smart, and also really, really dumb\n\nGullibility is the biggest unsolved problem\n\nCode may be the best application\n\nThe ethics of this space remain diabolically complex\n\nMy blog in 2023\n\nHere’s the sequel to this post: Things we learned about LLMs in 2024.\n\nLarge Language

### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [11]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [12]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [13]:
from ragas.testset.graph import Node, NodeType

for doc in docs:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 2, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [14]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/2 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/12 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/20 [00:00<?, ?it/s]

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 11, relationships: 29)

We can save and load our knowledge graphs as follows.

In [15]:
kg.save("ai_across_years_kg.json")
ai_across_years_kg = KnowledgeGraph.load("ai_across_years_kg.json")
ai_across_years_kg

KnowledgeGraph(nodes: 11, relationships: 29)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [16]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=ai_across_years_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [17]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

#### ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.

1. Single hop query requires a single retrieval
2. Multi-hop requires multiple pieces of context to be answered
3. "Abstract" requires reasoning about context, rather than just extracting 
   info from the query.

#### ❗ Question #1:

These are best understood by looking at 

   a. single vs. multi-hop, 
   b. specific vs. abstract split.

To a certain degree, the distinction appears to be dependent on the source of context.
That is, one can imagine that for one context, a certain query is abstract (resp. multi-hop)
but for another it is specific (resp. single-hop).
One can likely extend a context by Q&A against multi-hop and abstract queries to
generate a new context where the original query is single-hop/specific.

##### Single- vs. Multi-hop
A single-hop query requires a single query against the context (vector) database,
even if that query returns multiple documents.
A multi-hop query requires multiple retrievals, such that each additional retrieval
is conditional on results of the first retrieval.

Here are examples of two such queries:

Type | Example | Retrieval behavior |
| --- | --- | --- |
Single-hop | "What year did Albert Einstein win the Nobel Prize?" | One query into vector DB gets back a document mentioning Einstein and the Nobel Prize in 1921. No need to retrieve again. |
Multi-hop | "Which university employed the Nobel Prize winner in physics from 1921?" | 1st retrieval: Find that Einstein won Nobel in 1921 → 2nd retrieval: Query about Einstein’s employment history. |

##### Specific vs. Abstract

Answers for specific queries are directly retrievable from data.
Answers for abstract queries require reasoning, summarization, or inference about
the data given in the retrieved context.

Type | Example
| --- | --- |
Specific Query | "What is the capital of France?" ( Document says: "Paris is the capital of France.")
Abstract Query | "Why is Paris considered a center of art and culture?" ( Multiple facts about history, museums, famous artists, etc. must be inferred together.)


Finally, we can use our `TestSetGenerator` to generate our testset!

In [18]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating personas:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,What is the significance of Anthropic in the d...,[The ethics of this space remain diabolically ...,"According to the provided context, Anthropic h...",single_hop_specifc_query_synthesizer
1,What does the term Microsoft mean in the conte...,"[and software engineer, LLMs are infuriating. ...",The context discusses the challenges and frust...,single_hop_specifc_query_synthesizer
2,Considering the advancements in large language...,[Simon Willison’s Weblog Subscribe Stuff we fi...,"According to Simon Willison’s weblog, 2023 was...",single_hop_specifc_query_synthesizer
3,Is it OK to train models on people's content w...,[the document includes some of the clearest ex...,The context discusses ethical questions relate...,single_hop_specifc_query_synthesizer
4,"As an AI researcher and developer, how has the...",[Everything tagged “llms” on my blog in 2024 T...,"In 2024, the GPT-4 barrier was comprehensively...",single_hop_specifc_query_synthesizer
5,"How do organizations like OpenAI, Anthropic, G...",[<1-hop>\n\nThe ethics of this space remain di...,The context highlights that multiple organizat...,multi_hop_abstract_query_synthesizer
6,How do the challenges in model development rel...,"[<1-hop>\n\nand software engineer, LLMs are in...",The context highlights that LLMs are complex a...,multi_hop_abstract_query_synthesizer
7,How do the recent advancements in AI-generated...,"[<1-hop>\n\nNotebookLM, released in September,...","Recent advancements, including Claude Artifact...",multi_hop_abstract_query_synthesizer
8,How does Meta's development of models with inc...,[<1-hop>\n\nEverything tagged “llms” on my blo...,Meta's development of models with increased co...,multi_hop_specific_query_synthesizer
9,How have the recent price reductions and multi...,[<1-hop>\n\ncollapse in the cost of running a ...,"The recent price reductions, driven by increas...",multi_hop_specific_query_synthesizer


### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [20]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/2 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/12 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/18 [00:00<?, ?it/s]

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [21]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,What organization is included in the list of o...,[The ethics of this space remain diabolically ...,"According to the provided context, organizatio...",single_hop_specifc_query_synthesizer
1,What are Large Language Modles?,[Simon Willison’s Weblog Subscribe Stuff we fi...,Large Language Models are the latest and most ...,single_hop_specifc_query_synthesizer
2,"What are LLMs, and how are they impacting AI d...",[the document includes some of the clearest ex...,The context explains that large language model...,single_hop_specifc_query_synthesizer
3,What does Meta refer to in the context of larg...,[Everything tagged “llms” on my blog in 2024 T...,"In the context of large language models, Meta ...",single_hop_specifc_query_synthesizer
4,Hw impact of comp and efficieny on AI pricng a...,[<1-hop>\n\nEverything tagged “llms” on my blo...,The context explains that increased competitio...,multi_hop_abstract_query_synthesizer
5,How do model size and parameters relate to con...,[<1-hop>\n\na relevant paper Training Large La...,"The context highlights that DeepSeek v3, a 685...",multi_hop_abstract_query_synthesizer
6,How do recent advancements in multi-modal LLMs...,[<1-hop>\n\ncollapse in the cost of running a ...,Recent advancements in multi-modal LLMs like G...,multi_hop_abstract_query_synthesizer
7,how model size and data quality affect model c...,[<1-hop>\n\na relevant paper Training Large La...,"DeepSeek v3, a 685B parameter model, was train...",multi_hop_abstract_query_synthesizer
8,Meta still no beat GPT-4 how Meta's meta model...,[<1-hop>\n\nEverything tagged “llms” on my blo...,"The context shows that despite Meta's efforts,...",multi_hop_specific_query_synthesizer
9,How do the explanations of what LLMs are and h...,[<1-hop>\n\nthe document includes some of the ...,The first segment provides clear explanations ...,multi_hop_specific_query_synthesizer


In [22]:
dataset.get_sample_type()

ragas.testset.synthesizers.testset_schema.TestsetSample

We'll need to provide our LangSmith API key, and set tracing to "true".

# 🤝 BREAKOUT ROOM #2

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [ ]:
from langsmith import Client

client = Client()

dataset_name = "State of AI Across the Years!"
#client.delete_dataset(dataset_name=dataset_name)
langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="State of AI Across the Years!"
)

We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [25]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

In [26]:
dataset

Testset(samples=[TestsetSample(eval_sample=SingleTurnSample(user_input='What organization is included in the list of organizations that have produced better-than-GPT-3 class models, alongside EleutherAI?', retrieved_contexts=None, reference_contexts=['The ethics of this space remain diabolically complex My blog in 2023 Here’s the sequel to this post: Things we learned about LLMs in 2024. Large Language Models In the past 24-36 months, our species has discovered that you can take a GIANT corpus of text, run it through a pile of GPUs, and use it to create a fascinating new kind of software. LLMs can do a lot of things. They can answer questions, summarize documents, translate from one language to another, extract information and even write surprisingly competent code. They can also help you cheat at your homework, generate unlimited streams of fake content and be used for all manner of nefarious purposes. So far, I think they’re a net positive. I’ve used them on a personal level to impro

## Basic RAG Chain

Time for some RAG!


In [27]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [28]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [29]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [30]:
from langchain_community.vectorstores import Qdrant

vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="State of AI"
)

In [31]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [32]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

For our LLM, we will be using TogetherAI's endpoints as well!

We're going to be using Meta Llama 3.1 70B Instruct Turbo - a powerful model which should get us powerful results!

In [33]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

Finally, we can set-up our RAG LCEL chain!

In [34]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain.schema import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [35]:
rag_chain.invoke({"question" : "What are Agents?"})

'Based on the context provided, "agents" is an infuriatingly vague term with no single, clear, and widely understood meaning. The term typically refers to AI systems that can go away and act on your behalf, often likened to the travel agent model, or large language models (LLMs) given access to tools which they can use iteratively to solve problems. However, the term is used inconsistently, with dozens of possible definitions, and "agents" as practical, robust systems have not yet truly materialized in production. Additionally, the concept of agents is often tied to the idea of autonomy and sometimes connected to the achievement of AGI (Artificial General Intelligence), given challenges like gullibility in current models. In summary, "agents" are AI systems intended to act independently to assist users, but the concept remains vague, underdefined, and currently more of a "coming soon" idea than a realized technology.'

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4.1 as our evaluation LLM for our base Evaluators.

In [36]:
eval_llm = ChatOpenAI(model="gpt-4.1-mini")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [37]:
from langsmith.evaluation import LangChainStringEvaluator, evaluate

qa_evaluator = LangChainStringEvaluator("qa", config={"llm" : eval_llm})

labeled_helpfulness_evaluator = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": {
            "helpfulness": (
                "Is this submission helpful to the user,"
                " taking into account the correct reference answer?"
            )
        },
        "llm" : eval_llm
    },
    prepare_data=lambda run, example: {
        "prediction": run.outputs["output"],
        "reference": example.outputs["answer"],
        "input": example.inputs["question"],
    }
)

dope_or_nope_evaluator = LangChainStringEvaluator(
    "criteria",
    config={
        "criteria": {
            "dopeness": "Is this submission dope, lit, or cool?",
        },
        "llm" : eval_llm
    }
)

#### 🏗️ Activity #2:

Highlight what each evaluator is evaluating.

- `qa_evaluator`:
    Evaluates how well an answer matches the question.
    It uses an LLM as the evaluator, and prompts it with the query,
    the reference (ground-truth) answer, and the model output.
    The LLM evaluator is supposed to judge how close the model answer is to the
    query and the ground-truth answer.

- `labeled_helpfulness_evaluator`: This is a `labeled_criteria` evaluator 
    that judges the correspondence of the model answer along "helpfulness" axis
    to the query in reference to the reference answer. 
    The correspondence is measured by the LLM. An added feature (not used here)
    is that manual/human labels can be associated with the reference answers
    to further inform the "grade scale" of "helpfulness" for the LLM.

- `dope_or_nope_evaluator`:
    This is an (unlabeled) `criteria` evaluator 
    that judges the correspondence of the model answer along "dopeness" axis
    to the query **without** reference to the reference answer (and therefore
    without reference to any human-defined labels either). 
    The correspondence is measured by the LLM.




## LangSmith Evaluation

In [38]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dope_or_nope_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'bold-music-86' at:
https://smith.langchain.com/o/d5aca770-e410-428e-97f9-497517327fbd/datasets/039731fa-83f6-4507-a2c1-48838c477852/compare?selectedSessions=9ce88976-9fba-4104-b73c-143e36bef874




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,How do LLMs work and what are the main challen...,"Based on the provided context, here is what ca...",None,The document includes clear explanations of wh...,1,1,1,8.211822,5989cd9a-7336-4061-b497-66e07e897ecf,397a7a7b-f5cc-4cbe-bdbe-51df61d8f026
1,How does Anthropic's development of Claude Art...,"Based on the provided context, Anthropic's dev...",None,"Anthropic's development of Claude Artifacts, a...",1,1,1,4.804016,bc3bcb6d-a2e6-4075-b699-d4461c0eba6d,9daed97f-d235-4278-9813-0ae6e1bd485d
2,How do the explanations of what LLMs are and h...,The first segment explains that LLMs are compl...,None,The first segment provides clear explanations ...,0,1,0,13.999038,a9d796a3-b1b0-4601-8c4b-b5ab36f27110,e021c7f0-ce39-420d-8a29-560b4812f4a9
3,Meta still no beat GPT-4 how Meta's meta model...,"Based on the provided context, Meta is one of ...",None,"The context shows that despite Meta's efforts,...",0,0,1,6.602942,11451d25-405d-4459-b007-8c53e915e7ea,7bbfe81f-a2f8-454c-83f6-cc616635da45
4,how model size and data quality affect model c...,I don't know.,None,"DeepSeek v3, a 685B parameter model, was train...",0,0,0,1.658174,b3ecbbd4-bbb9-4308-87fe-e4d42452c7e3,6e3fa8ea-d810-4604-8a66-be66196d5dec
5,How do recent advancements in multi-modal LLMs...,I don't know.,None,Recent advancements in multi-modal LLMs like G...,0,0,0,1.540156,48fe2bb5-6c2d-4b06-bf4e-f39976779996,f8c5431e-6870-426d-ae1f-76facaf66993
6,How do model size and parameters relate to con...,I don't know.,None,"The context highlights that DeepSeek v3, a 685...",0,0,0,2.655381,e8daf5b8-93db-4500-9aa1-27404ecbab42,3330590e-97ba-4e78-8e6c-2352805fd5d7
7,Hw impact of comp and efficieny on AI pricng a...,The impact of competition and efficiency on AI...,None,The context explains that increased competitio...,1,1,0,4.758769,9b175ea2-fbaf-4148-8c94-9c54dd27b8bb,f73fc6ea-e6c2-4fea-9998-2720c67ced6b
8,What does Meta refer to in the context of larg...,Meta refers to an organization that develops l...,None,"In the context of large language models, Meta ...",1,1,1,5.083996,7bc0957a-77ca-4040-835a-ed93ef4089e1,0d7ddb1e-bd9f-4c21-9976-68381f45783c
9,"What are LLMs, and how are they impacting AI d...","LLMs, or Large Language Models, are a kind of ...",None,The context explains that large language model...,1,1,1,5.281932,e2a33899-475c-48bf-a6ed-d033d1fb4ab3,7fc0d9fe-6034-41aa-b71e-146d5d3ae1f3


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [39]:
DOPE_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

You must answer the questions in a dope way, be cool!

Context: {context}
Question: {question}
"""

dope_rag_prompt = ChatPromptTemplate.from_template(DOPE_RAG_PROMPT)

In [40]:
rag_documents = docs

In [41]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

#### ❓Question #2:

Why would modifying our chunk size modify the performance of our application?

#### ❗Answer #2:

The chunk size is what determines the length of the documents retrieved
by the R part of RAG. Longer chunk size means longer piece of context.

In this manner, single-step RAG that retrieves longer chunks has a better
chance of providing a comprehensive answer than one with shorter chunks.

At the same time, longer chunks may "mask" the (in)ability of a multi-step RAG.
If we write a query that is expected to be a two-step evaluation query,
but the chunk size is so long that both "bits" of context needed to answer the
question end up being in the same retrieved document, the RAG may end up performing
very well even if its multi-step ability is not that good.

In terms of speed, though, having large chunks retrieved into the context may
grow the Augmented prompt for the RAG be longer than the context of the LLM
allows, or it may require very long processing times to answer even simple questions.
This is especially true when naive chunking is used (like here) and not
one of the compressing chunkers.


In [55]:
from langchain_openai import OpenAIEmbeddings

print(f"Old embedding model: {generator_embeddings.embeddings.model}")
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
print(f"New embedding model: {embeddings.model}")

Old embedding model: text-embedding-ada-002
New embedding model: text-embedding-3-large


In [58]:
print( embeddings.dimensions )

None


#### ❓Question #3:

Why would modifying our embedding model modify the performance of our application?

#### ❗Answer #3:

The purpose of an embedding model is to preserve the meaning of a chunk while 
compressing it to an embedding representation. If the embedding model is 
performing poorly, it may map two semantically-unrelated chunks into
a region of the embedding space where they become near-neighbors.

When the embedding space is queried, the query itself gets mapped into the 
embedding space. A bad embedding model may map it into vicinity of 
chunks that are unrelated to it. So the retrieved context ends up being 
unhelpful for answering the original query.

On the other hand, if our context documents are semantically simple, but large 
in the number of chunks (think a document consisting only of words 'hot' and 'cold' repeated many times) the
target dimension of the embedding space (1536) may be much larger than needed.
This ends up costing compute time without added benefit.


In [43]:
vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="AI Across Years (Augmented)"
)

In [44]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [45]:
dope_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | dope_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [46]:
dope_rag_chain.invoke({"question" : "what are Agents?"})

'Yo, so here’s the lowdown on Agents straight from the context: “Agents” is one of those buzzwords that’s kinda fuzzy as heck—people toss it around but rarely nail down a single, clear meaning. Some folks see agents as AI stepping out to *actually act* on your behalf, like a savvy travel agent booking your flights. Others think of ‘agents’ as LLMs hooked up with tools, hustling through loops to crack problems.\n\nBut real talk? Agents still feel like a “coming soon” vibe, because the big hurdle is gullibility—these models can’t always tell fact from fiction, which makes trusting them with decisions a tricky game. So until they get that “truth detector” upgrade (maybe via AGI someday), the dream of agents doing their thang reliably is still a bit out of reach.\n\nIn short: Agents = AI trying to act *for* you, but with a massive “work in progress” stamp until they figure out how to not get played by misinformation and prompt hacks. Keep your eyes peeled, but the magic ain’t fully here ye

Finally, we can evaluate the new chain on the same test set!

In [47]:
evaluate(
    dope_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dope_or_nope_evaluator
    ],
    metadata={"revision_id": "dope_chain"},
)

View the evaluation results for experiment: 'slight-comfort-48' at:
https://smith.langchain.com/o/d5aca770-e410-428e-97f9-497517327fbd/datasets/039731fa-83f6-4507-a2c1-48838c477852/compare?selectedSessions=44841662-65b5-428d-9c89-5537226b5b02




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,How do LLMs work and what are the main challen...,"Alright, let me break it down fresh and fly fo...",None,The document includes clear explanations of wh...,1,1,1,7.248891,5989cd9a-7336-4061-b497-66e07e897ecf,80449ab5-21ad-49ef-b971-21b33741a813
1,How does Anthropic's development of Claude Art...,"Alright, here’s the lowdown, cool style: Anthr...",None,"Anthropic's development of Claude Artifacts, a...",1,1,1,5.983047,bc3bcb6d-a2e6-4075-b699-d4461c0eba6d,77597225-7b89-4c32-a058-553b7d8d1dd1
2,How do the explanations of what LLMs are and h...,"Alright, here’s the lowdown in a fresh flow. T...",None,The first segment provides clear explanations ...,0,0,1,8.806254,a9d796a3-b1b0-4601-8c4b-b5ab36f27110,06613f3f-62bb-400b-9600-e2c5d5e26290
3,Meta still no beat GPT-4 how Meta's meta model...,"Yo, based on the vibe from the context, Meta’s...",None,"The context shows that despite Meta's efforts,...",0,0,1,2.674569,11451d25-405d-4459-b007-8c53e915e7ea,6089159d-de8b-48d4-bad8-29523285ba32
4,how model size and data quality affect model c...,"Alright, here’s the lowdown straight from the ...",None,"DeepSeek v3, a 685B parameter model, was train...",1,1,1,5.316299,b3ecbbd4-bbb9-4308-87fe-e4d42452c7e3,7c4cdc24-9a93-48d1-9eab-b573a06f2a27
5,How do recent advancements in multi-modal LLMs...,"Yo, based on the vibe from the context, here’s...",None,Recent advancements in multi-modal LLMs like G...,1,1,1,15.954632,48fe2bb5-6c2d-4b06-bf4e-f39976779996,d09f7318-2504-4148-8f62-a0d149923439
6,How do model size and parameters relate to con...,"Alright, here’s the lowdown straight from the ...",None,"The context highlights that DeepSeek v3, a 685...",1,1,1,4.212033,e8daf5b8-93db-4500-9aa1-27404ecbab42,2b194796-111f-48a7-b0f0-b4b3ed434977
7,Hw impact of comp and efficieny on AI pricng a...,"Yo, here’s the lowdown: Competition and effici...",None,The context explains that increased competitio...,1,1,1,2.636694,9b175ea2-fbaf-4148-8c94-9c54dd27b8bb,a5f2033e-5b23-4afd-a276-80c19f7adfa8
8,What does Meta refer to in the context of larg...,"Yo, Meta in this LLM game is all about their L...",None,"In the context of large language models, Meta ...",0,0,1,2.775573,7bc0957a-77ca-4040-835a-ed93ef4089e1,a87ff155-2e79-47f1-8fda-c3ccf44d5316
9,"What are LLMs, and how are they impacting AI d...","Alright, here’s the lowdown on LLMs straight f...",None,The context explains that large language model...,1,1,1,10.032935,e2a33899-475c-48bf-a6ed-d033d1fb4ab3,55f43ebb-f0bb-49db-a729-165e5e882fc0


#### 🏗️ Activity #3:

Provide a screenshot of the difference between the two chains, and explain why you believe certain metrics changed in certain ways.

Remember, these were the changes:

- Include a "dope" prompt augmentation ("You must answer the questions in a dope way, be cool!")
- Use larger chunks: 500 ➡️ 1000
- Improve the retriever model: `text-embedding-ada-002` ➡️ `text-embedding-3-large`


![Comparison](media/compare-eval.png)

![Comparison](media/compare-answers.png)